In [1]:
import cv2
import os
import math

In [ ]:
video_paths = {
    "gopro": {'path':"./input/g_joanna.MP4",
    'start_offset': 30 ,
    "end_offset" : 5
    }
}

In [ ]:
video_paths

In [4]:
#  for each video ( participation ) we need to manually enter start and end offset 
# ( base on secound )

In [ ]:
# ==========================================
# OVERVIEW: Initializes video captures, handles missing files or invalid 
# framerates, and calculates usable duration using start/end offsets.
#
# VARIABLES TO CHANGE:
# - `config`: Dict containing "path", "start_offset", and "end_offset".
# - `fps = 30.0`: Fallback framerate if OpenCV fails to read it.
# ==========================================

import cv2

def setup_video_capture_with_end(config):
    info = {}
    for name, data in config.items():
        cap = cv2.VideoCapture(data["path"])
        
        # 1. Check if video opened successfully
        if not cap.isOpened():
            print(f"⚠️ Warning: Could not open video at path '{data['path']}' for '{name}'. Skipping.")
            continue

        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # 2. Prevent division by zero if FPS couldn't be read
        if fps <= 0:
            print(f"⚠️ Warning: Could not read FPS for '{name}' (returned {fps}). Defaulting to 30.0 FPS.")
            fps = 30.0  # Or handle as an error depending on your pipeline requirements

        total_duration = total_frames / fps

        actual_end_time = total_duration - data.get("end_offset", 0.0)
        usable_duration = actual_end_time - data.get("start_offset", 0.0)

        info[name] = {
            "cap": cap,
            "fps": fps,
            "start_offset": data.get("start_offset", 0.0),
            "usable_duration": usable_duration
        }
    return info

**Context:  check to confirm the video path and crop boundaries are formatted correctly.**

In [10]:
%%time
info = setup_video_capture_with_end(video_paths)
info

CPU times: user 16.2 ms, sys: 0 ns, total: 16.2 ms
Wall time: 12.4 ms


{'gopro': {'cap': < cv2.VideoCapture 0x7f731f3bb6b0>,
  'fps': 25.0,
  'start_offset': 30,
  'usable_duration': 310.84}}

In [ ]:
# ==========================================
# OVERVIEW: Synchronizes multiple video streams into fixed-time chunks. 
# Calculates the exact starting frame for each camera per chunk based on 
# time offsets and verifies frame availability.
#
# VARIABLES TO CHANGE:
# - `chunk_duration=3`: Length of each time chunk in seconds.
# - `chunk_idx % 20 == 0`: The interval for printing progress updates (currently every 20 chunks).
# ==========================================

def process_chunks(video_info, chunk_duration=3):
    # List to store the final results of all chunks
    all_chunks_results = []

    min_usable_duration = min(v["usable_duration"] for v in video_info.values())
    total_chunks = math.floor(min_usable_duration / chunk_duration)

    print(f"Starting processing for {total_chunks} chunks...")

    for chunk_idx in range(total_chunks):
        exp_time_start = chunk_idx * chunk_duration
        exp_time_end = (chunk_idx + 1) * chunk_duration

        chunk_record = {
            "chunk_id": chunk_idx,
            "time": f"{exp_time_start}-{exp_time_end}",
            "cameras_data": {},
            "objects": [],
        }

        for cam_name, data in video_info.items():
            target_frame = int((data["start_offset"] + exp_time_start) * data["fps"])
            data["cap"].set(cv2.CAP_PROP_POS_FRAMES, target_frame)

            ret, frame = data["cap"].read()
            if ret:
                chunk_record["cameras_data"][cam_name] = {
                    "processed_frame": target_frame,
                    "status": "success"
                }
            else:
                chunk_record["cameras_data"][cam_name] = {"status": "failed"}

        # Add this chunk record to the main list
        all_chunks_results.append(chunk_record)

        if chunk_idx % 20 == 0:
            print(f"Chunk {chunk_idx} added to pipeline.")

    print("Processing complete.")
    return all_chunks_results

**Slices video into synchronized 3-second blocks and extracts a single snapshot frame from the start of each block.**
**Output A list of chunk dictionaries containing timing metadata, frame indices, and empty placeholders for future AI results.**


In [8]:
# cv2 format to the remove the offsets and turn into to 3 scound chunk 

In [11]:
%%time
processed_data = process_chunks(info)

Starting processing for 103 chunks...
Chunk 0 added to pipeline.
Chunk 20 added to pipeline.
Chunk 40 added to pipeline.
Chunk 60 added to pipeline.
Chunk 80 added to pipeline.
Chunk 100 added to pipeline.
Processing complete.
CPU times: user 5min 14s, sys: 1.68 s, total: 5min 16s
Wall time: 1min 2s


In [12]:
processed_data[:3]

[{'chunk_id': 0,
  'time': '0-3',
  'cameras_data': {'gopro': {'processed_frame': 750, 'status': 'success'}},
  'objects': []},
 {'chunk_id': 1,
  'time': '3-6',
  'cameras_data': {'gopro': {'processed_frame': 825, 'status': 'success'}},
  'objects': []},
 {'chunk_id': 2,
  'time': '6-9',
  'cameras_data': {'gopro': {'processed_frame': 900, 'status': 'success'}},
  'objects': []}]

In [ ]:
# ==========================================
# OVERVIEW: Utility function to save or load JSON data. 
# Automatically appends the .json extension and handles UTF-8 encoding.
#
# VARIABLES TO CHANGE:
# - `path_file = './' + path + '.json'`: Base directory path. You can swap 
#   this with the commented-out Google Drive path if running in Colab.
# - `indent=4`: JSON formatting indentation when saving files.
# - Function parameters: `path` (filename), `save` (toggle read/write mode), 
#   and `config` (the data dictionary to save).
# ==========================================

import os
import json

def save_load(path:str,save=False,config = None):
    path_file = './' + path + '.json'
    processed_data = {}

    # path_file = "/content/drive/MyDrive/data/" + path + '.json'
    

    if save:
        
        with open(path_file,'w',encoding='utf-8') as f :
            json.dump(config,f,ensure_ascii=False, indent=4)
    else:
        if os.path.exists(path_file):
            with open(path_file, 'r', encoding='utf-8') as f :
                processed_data = json.load(f)
                return processed_data

**Save and Load Json object**

In [11]:
!pip install ultralytics --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [Errno 16] Device or resource busy: '.nfs00000000045c841d000009ba'



In [14]:
# change 
# from ultralytics import YOLO
from ultralytics import YOLOWorld
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLOWorld('yolov8l-worldv2.pt').to(device)
# if you want to use smaller model the L model 90m size 
# model = YOLO('yolov8s-world.pt').to(device)

**install YOLO for obj detection**

In [15]:
custom_classes = [
    "measuring tape", "tape measure", "yellow tape measure", "ruler",
    "paper shredder", "shredder machine", "trash can",'shredder' 
    "scissors", "paper", "pen", "pencil", "marker", 
    "cardboard box", "carton", "box", "package", 
    "keyboard", "monitor", "laptop","yellow measuring tape", "measuring tape", "tape measure", 
    "scissors", "metal scissors", "shears", 
    "paper shredder", "shredder machine",
    "keyboard", "computer monitor", "pen", "paper", "box"
]

model.set_classes(custom_classes)

**set custom classes for our recoding , need to later set sepecif set of class for each work**

In [ ]:
# ==========================================
# OVERVIEW: Runs object detection on extracted frames from the 'gopro' 
# camera stream. Extracts unique object tags for each temporal chunk 
# and merges them back into the structured data pipeline.
#
# VARIABLES TO CHANGE:
# - `conf_val=0.12`: Confidence threshold for the detection model.
# - `cam_name = "gopro"`: Hardcoded target camera name to process.
# - Print frequency: `chunk["chunk_id"] % 20 == 0` controls log output.
# - Note: `device` and `custom_classes` must be defined globally.
# ==========================================

def process_objects_gopro(processed_data, video_info, model, conf_val=0.12):
    for chunk in processed_data:
        detected_names = []
        
        cam_name = "gopro"
        if cam_name in chunk["cameras_data"] and chunk["cameras_data"][cam_name]["status"] == "success":
            
            meta = chunk["cameras_data"][cam_name]
            cap = video_info[cam_name]["cap"]
            cap.set(cv2.CAP_PROP_POS_FRAMES, meta["processed_frame"])
            ret, frame = cap.read()
            
            if ret:

                results = model.predict(frame, conf=conf_val, device=device, verbose=False)
                
                for r in results:
                    for c in r.boxes.cls:
                        obj_name = custom_classes[int(c)]
                        detected_names.append(obj_name)
        
        # Save to the active objects list (removes duplicates using set)
        
        chunk["objects"] = list(set(detected_names))
        
        if chunk["chunk_id"] % 20 == 0:
            print(f"✅ Chunk {chunk['chunk_id']} analyzed.")
            
    return processed_data

**Runs YOLO object detection on the extracted GoPro frame for each 3-second chunk.   Seeks to the target frame, reads it via OpenCV, predicts objects using the YOLO model (with a default confidence threshold of 0.12), maps numerical predictions back to custom class names, and saves a deduplicated list of detected item**

In [17]:
%%time
processed_data_ob_de = process_objects_gopro(processed_data,info,model)

✅ Chunk 0 analyzed.
✅ Chunk 20 analyzed.
✅ Chunk 40 analyzed.
✅ Chunk 60 analyzed.
✅ Chunk 80 analyzed.
✅ Chunk 100 analyzed.
CPU times: user 5min 15s, sys: 1.38 s, total: 5min 17s
Wall time: 1min 2s


**use YOLO with custom class to analyis only gopro cam to detect object and save on that same structure**


In [18]:
processed_data_ob_de[69:73]

[{'chunk_id': 69,
  'time': '207-210',
  'cameras_data': {'gopro': {'processed_frame': 5925, 'status': 'success'}},
  'objects': ['metal scissors',
   'laptop',
   'pencil',
   'keyboard',
   'yellow measuring tape',
   'box',
   'scissors',
   'computer monitor'],
  'action_probs': {}},
 {'chunk_id': 70,
  'time': '210-213',
  'cameras_data': {'gopro': {'processed_frame': 6000, 'status': 'success'}},
  'objects': ['metal scissors',
   'pencil',
   'keyboard',
   'marker',
   'pen',
   'computer monitor'],
  'action_probs': {}},
 {'chunk_id': 71,
  'time': '213-216',
  'cameras_data': {'gopro': {'processed_frame': 6075, 'status': 'success'}},
  'objects': ['box', 'laptop', 'keyboard'],
  'action_probs': {}},
 {'chunk_id': 72,
  'time': '216-219',
  'cameras_data': {'gopro': {'processed_frame': 6150, 'status': 'success'}},
  'objects': ['cardboard box', 'box'],
  'action_probs': {}}]

In [ ]:
# ==========================================
# OVERVIEW: Standardizes raw object tags into unified, uppercase 
# categories (e.g., "tape" -> "MEASURING_TAPE") for consistency.
#
# VARIABLES TO CHANGE:
# - `processed_data`: The list of chunk dictionaries.
# - Mapping rules: The keywords and target categories in the `if/elif` block.
# ==========================================

def normalize_chunk_data(processed_data):
    for chunk in processed_data:
        raw_list = chunk.get("objects", [])
        normalized = set()
        
        for item in raw_list:
            item = item.lower()
            # منطق مپ کردن کلمات
            if any(x in item for x in ["tape", "measuring", "ruler"]):
                normalized.add("MEASURING_TAPE")
            elif any(x in item for x in ["scissors", "shears", "metal"]):
                normalized.add("SCISSORS")
            elif "shredder" in item:
                normalized.add("SHREDDER")
            elif "keyboard" in item:
                normalized.add("KEYBOARD")
            elif "monitor" in item or "screen" in item:
                normalized.add("MONITOR")
            elif "pen" in item or "marker" in item:
                normalized.add("PEN")
            elif "paper" in item:
                normalized.add("PAPER")
            elif "box" in item or "cardboard" in item:
                normalized.add("BOX")
            else:
                normalized.add(item.upper())
        
        chunk["objects"] = list(normalized)
    return processed_data

**the layer normoliser custoom class to one things**

In [20]:
norm_data = normalize_chunk_data(processed_data_ob_de)

In [21]:
import copy 
norm_data_deep_copy = copy.deepcopy(norm_data)

In [22]:
norm_data_deep_copy[30:34]

[{'chunk_id': 30,
  'time': '90-93',
  'cameras_data': {'gopro': {'processed_frame': 3000, 'status': 'success'}},
  'objects': ['LAPTOP', 'PEN', 'KEYBOARD'],
  'action_probs': {}},
 {'chunk_id': 31,
  'time': '93-96',
  'cameras_data': {'gopro': {'processed_frame': 3075, 'status': 'success'}},
  'objects': ['LAPTOP', 'PEN', 'KEYBOARD'],
  'action_probs': {}},
 {'chunk_id': 32,
  'time': '96-99',
  'cameras_data': {'gopro': {'processed_frame': 3150, 'status': 'success'}},
  'objects': ['PEN', 'MEASURING_TAPE', 'MONITOR', 'KEYBOARD'],
  'action_probs': {}},
 {'chunk_id': 33,
  'time': '99-102',
  'cameras_data': {'gopro': {'processed_frame': 3225, 'status': 'success'}},
  'objects': ['LAPTOP', 'BOX', 'KEYBOARD'],
  'action_probs': {}}]

In [ ]:
save_load(path='./output/video_obj_detected',save=True,config=norm_data_deep_copy)